In [1]:
%pip install scikit-learn==1.9.1 joblib tabulate

Note: you may need to restart the kernel to use updated packages.


Step 1: Dataset choice

Minimum Feature Size: 12 
Minimum Instance Size: 500

In [3]:
import pandas as pd
df = pd.read_csv("ai4i2020.csv")
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Read our saved local dataset
df = pd.read_csv("ai4i2020.csv")

# 2. Drop serial numbers and sub-failure codes
drop_cols = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df_clean = df.drop(columns=drop_cols)

# 3. Convert machine type letters ('L', 'M', 'H') to numbers (0, 1, 2)
df_clean['Type'] = df_clean['Type'].astype('category').cat.codes

# 4. Separate inputs (X) from the target diagnosis (y)
X = df_clean.drop(columns=['Machine failure'])
y = df_clean['Machine failure']

# 5. Split: 80% to train, 20% held out to test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 6. Create the 'model' directory
os.makedirs("model", exist_ok=True)

# 7. Save test data (must include all inputs + target column 'Machine failure')
test_data = pd.concat([X_test, y_test], axis=1)
test_data.to_csv("test_data.csv", index=False)
print("1. test_data.csv saved successfully!")

# 8. Scale numerical features and save scaler for Streamlit app
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "model/scaler.pkl")
print("2. model/scaler.pkl saved successfully!")

1. test_data.csv saved successfully!
2. model/scaler.pkl saved successfully!


Step 2: Machine Learning Classification models and Evaluation metrics

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbor Classifier
4. Naive Bayes Classifier - Gaussian or Multinomial
5. Support Vector Machine
   
For each of the models above, calculate the following evaluation metrics:
1. Accuracy
2. AUC Score
3. Precision
4. Recall
5. F1 Score
6. Matthews Correlation Coefficient (MCC Score)

In [6]:
# 1. Import the 5 classifiers and the 6 evaluation metrics
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)
import joblib
import pandas as pd

# 2. Put the 5 models in a dictionary
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=42)
}

# 3. Create an empty list to store scores
comparison_records = []

# 4. Train each model, save it, and test it
for name, model in models.items():
    # A. Teach the model
    model.fit(X_train_scaled, y_train)
    
    # B. Save model brain to the model folder
    clean_filename = name.lower().replace(" ", "_")
    joblib.dump(model, f"model/{clean_filename}.pkl")
    
    # C. Test on the unseen 20% test data
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # D. Calculate the 6 required metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # E. Append row to our record list
    comparison_records.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

# 5. Display the comparison table
results_table = pd.DataFrame(comparison_records)
print(results_table.to_markdown(index=False))

/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


| ML Model Name          |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:-----------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression    |      0.968 | 0.9004 |      0.6667 |   0.1176 | 0.2    | 0.2712 |
| Decision Tree          |      0.973 | 0.9147 |      0.7692 |   0.2941 | 0.4255 | 0.4656 |
| KNN                    |      0.974 | 0.8292 |      0.8333 |   0.2941 | 0.4348 | 0.4861 |
| Naive Bayes            |      0.96  | 0.9053 |      0.2857 |   0.1176 | 0.1667 | 0.1655 |
| Support Vector Machine |      0.972 | 0.9453 |      0.875  |   0.2059 | 0.3333 | 0.4167 |
